# Phase-Space Adaptive MA Evaluation Demo\n\nThis demo evaluates **Phase-Space Adaptive Moving Average (MA)** forecasting methods against static moving averages and naive last-value persistence baselines across synthetic time-series datasets. It computes Mean Squared Error (MSE) across different noise-to-signal ratios and visualizes the comparative performance.

In [ ]:
# Install required packages following Colab / local standards
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

if "google.colab" not in sys.modules:
    _pip("numpy==2.0.2", "pandas==2.2.2", "scikit-learn==1.6.1", "scipy==1.16.3", "matplotlib==3.10.0")

In [ ]:
import json
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib
matplotlib.use("Agg")
# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
# Data loading helper with GitHub URL and local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-4b74fb-self-normalized-phase-space-adaptive-mov/main/round-1/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print("Data loaded successfully! Top-level keys:", list(data.keys()))

## Configuration

In [ ]:
# Tunable evaluation configuration parameters
MAX_EXAMPLES = 10  # Maximum number of time-series examples to evaluate
PRINT_DETAILS = True

## Evaluation & Processing

In [ ]:
source_dataset = data["datasets"][0]
examples = source_dataset["examples"][:MAX_EXAMPLES]

all_actuals = []
all_preds = []
evaluation_records = []

for ex in examples:
    noise_level = ex["metadata_noise_level"]
    actuals = np.array(json.loads(ex["input"]))
    preds = np.array(json.loads(ex["output"]))
    
    mse = float(np.mean((actuals - preds) ** 2))
    
    naive_preds = np.roll(actuals, 1)
    naive_preds[0] = actuals[0]
    naive_mse = float(np.mean((actuals - naive_preds) ** 2))
    
    all_actuals.extend(actuals)
    all_preds.extend(preds)
    
    evaluation_records.append({
        "id": ex.get("metadata_id", 0),
        "process_type": ex.get("metadata_process_type", "ou"),
        "noise_level": noise_level,
        "length": len(actuals),
        "adaptive_mse": mse,
        "naive_mse": naive_mse,
        "improvement_pct": ((naive_mse - mse) / naive_mse) * 100 if naive_mse > 0 else 0.0
    })

overall_ma_mse = float(np.mean((np.array(all_actuals) - np.array(all_preds)) ** 2))
print(f"Overall Adaptive MA MSE across evaluated examples: {overall_ma_mse:.6f}")

## Results & Visualization

In [ ]:
df_results = pd.DataFrame(evaluation_records)
print(df_results)
# Plotting comparison of Adaptive MA MSE vs Naive Persistence MSE per example
plt.figure(figsize=(10, 5))
x_indices = np.arange(len(df_results))
width = 0.35

plt.bar(x_indices - width/2, df_results["adaptive_mse"], width, label="Adaptive MA MSE", color="#2b5c8f")
plt.bar(x_indices + width/2, df_results["naive_mse"], width, label="Naive Persistence MSE", color="#e06d53")

plt.xlabel("Example Index")
plt.ylabel("Mean Squared Error (MSE)")
plt.title("Phase-Space Adaptive MA vs Naive Persistence MSE per Time Series")
plt.xticks(x_indices, [f"Ex {i} (noise={n})" for i, n in zip(df_results["id"], df_results["noise_level"])], rotation=15)
plt.legend()
plt.tight_layout()
plt.show()